In [1]:
import os

if not os.path.exists("COVID-ML"):
    !git clone https://github.com/vinimicius/COVID-ML.git
%cd COVID-ML

/content/COVID-ML


In [ ]:
!git checkout feat-random-forest

In [2]:
import os
import pandas as pd
from data_utils import run_pipeline

# Caminho do dado tratado
READY_DATA_PATH = 'data/data_ready.csv'

if os.path.exists(READY_DATA_PATH):
    print(f"✅ Carregando dados já tratados de: {READY_DATA_PATH}")
    df = pd.read_csv(READY_DATA_PATH)
else:
    print("🚀 Dados tratados não encontrados. Iniciando pipeline completa...")
    df = run_pipeline(export=True)

df.head()

✅ Carregando dados já tratados de: data/data_ready.csv


,F,M,idade,obito,asma,cardiopatia,diabetes,doenca_hematologica,doenca_hepatica,doenca_neurologica,doenca_renal,imunodepressao,obesidade,outros_fatores_de_risco,pneumopatia,puerpera,sindrome_de_down
0,0,1,0.738739,1,0,0,1,0,0,0,0,0,0,1,0,0,0
1,1,0,0.270270,1,0,0,1,0,0,0,0,0,1,0,0,0,0
2,0,1,0.288288,1,0,1,0,0,0,0,0,0,0,1,0,0,0
3,0,1,0.621622,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,0,1,0.702703,1,0,0,0,0,0,0,0,0,0,1,0,0,0


In [ ]:
# Definição clara para o leitor do notebook
AGE_GROUPS = {
    'jovem': (0, 30),
    'jovem-adulto': (20, 40),
    'adulto': (40, 60),
    'senior': (60, 80),
    'idoso': (80, 120)
}

In [ ]:
# Dicionário que servirá como nosso banco de dados de resultados
all_reports = {}

# Definimos o nosso grid aqui para ser consistente
meu_grid_final = {
    'n_estimators': [50, 100, 200, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [ ]:
from model_utils import (
    split_age_groups, 
    run_random_forest_pipeline, 
    get_model_predictions,
    display_group_report,
    export_visual_reports,
    save_model_assets
)

from sklearn.model_selection import train_test_split

# Separar os dados por faixa etária
groups = split_age_groups(df, AGE_GROUPS)


for name, group_df in groups.items():
    print(f"\n--- Processando Grupo: {name.upper()} ---")
    
    # 3. Preparação dos Dados
    X = group_df.drop(columns=['obito', 'idade_raw'], errors = 'ignore')
    y = group_df['obito']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # 4. Treino e Otimização (GridSearch via model_utils)
    best_model, best_params = run_random_forest_pipeline(X_train, y_train, name, param_grid=meu_grid_final)
    
    # 5. Predições
    y_pred, y_proba = get_model_predictions(best_model, X_test)

    # 6. EXIBIÇÃO E ARMAZENAMENTO (A parte que você perguntou)
    # A função printa o relatório na tela e guarda o DataFrame no dicionário
    all_reports[name] = display_group_report(y_test, y_pred, y_proba, name)

    # 7. Persistência (Salva o .pkl e o .json)
    save_model_assets(best_model, best_params, name)
    
    # 8. Exportação Silenciosa de Gráficos (Matriz de Confusão e Feature Importance)
    export_visual_reports(best_model, X_test, y_test, y_pred, name)
    
print("\n✅ Todos os grupos foram processados e armazenados em 'all_reports'!")

In [ ]:
# --- ANÁLISE GLOBAL ---
print(f"\n" + "="*60)
print(f"🌍 INICIANDO TREINAMENTO DO MODELO GLOBAL")
print(f"="*60)

# 1. Preparação dos Dados (Dataset Completo)
# Removemos apenas as colunas de alvo e a idade_raw (mantendo a idade escalonada)
X_global = df.drop(columns=['obito', 'idade_raw'], errors='ignore')
y_global = df['obito']

# 2. Split Global (Mantendo o random_state=42 para consistência estatística)
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_global, y_global, test_size=0.2, random_state=42, stratify=y_global
)

# 3. Treino e Otimização (Usando o mesmo Grid de Ouro)
name_global = "global"
best_model_g, best_params_g = run_random_forest_pipeline(
    X_train_g, y_train_g, name_global, param_grid=meu_grid_final
)

# 4. Predições
y_pred_g, y_proba_g = get_model_predictions(best_model_g, X_test_g)

# 5. Exibição e Armazenamento no nosso dicionário Master
# Agora o all_reports terá a chave 'global' ao lado dos grupos etários
all_reports[name_global] = display_group_report(y_test_g, y_pred_g, y_proba_g, name_global)

# 6. Persistência de Arquivos e Gráficos
save_model_assets(best_model_g, best_params_g, name_global)
export_visual_reports(best_model_g, X_test_g, y_test_g, y_pred_g, name_global)

print(f"\n✅ Modelo Global finalizado e armazenado!")